# <u>**Feature Engineering**</u>

The cleaned dataset is transformed into meaningful, model-ready features to support machine learning analysis of award patterns in the National Lottery Project Grants.

EDA shows the <code>award_amount</code> distribution is positively skewed and long‑tailed with meaningful variations between <code>main_dicipline</code>, <code>ace_area</code> and <code>strand</code>. 

A Decision Tree Classifier or Random Forest Classifier are suitable candidates for predicting award tiers. Unlike linear regression, which predicts continuous numerical values, these classifiers predict categorical outcomes. Tree-based models can capture non-linear relationships and do not require features to follow a normal distribution.

</b>

## <u>**Load Libraries and Data**</u>

The Pandas and NumPy libraries are imported, and the cleaned dataset is loaded using Pandas.

In [ ]:
# Import libraries 
import pandas as pd
import numpy as np

# Load DataFrame
df = pd.read_csv('/Users/admin/Documents/Code Institute/ace_project/data/clean/cleaned_data.csv')

# Create copy for feature engineering 
df_fe = df.copy()

</b>

## <u>**Check Data Integrity**</u>

A review of the cleaned dataset is made to confirm the structure, column types and statistical summary. 

In [2]:
# Review the DataFrame
df_fe.info()

# Summary statistics of the DataFrame
df_fe.describe()

<class 'pandas.DataFrame'>
RangeIndex: 5454 entries, 0 to 5453
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   award_amount           5454 non-null   int64
 1   award_date             5454 non-null   str  
 2   decision_month         5454 non-null   str  
 3   decision_quarter       5454 non-null   str  
 4   ace_area               5454 non-null   str  
 5   local_authority        5454 non-null   str  
 6   main_discipline        5454 non-null   str  
 7   strand                 5454 non-null   str  
 8   time_limited_priority  5454 non-null   str  
 9   award_year             5454 non-null   int64
dtypes: int64(2), str(8)
memory usage: 426.2 KB


,award_amount,award_year
count,5.454000e+03,5454.000000
mean,4.068841e+04,2024.189952
std,7.853711e+04,1.161237
min,0.000000e+00,2023.000000
25%,1.939175e+04,2023.000000
50%,2.940650e+04,2024.000000
75%,3.471500e+04,2025.000000
max,2.019149e+06,2026.000000


</b>

## <u>**Flag Zero Awards**</u>

EDA confirms that £0 values represent recorded funding decisions rather than data errors. A binary flag is created to identify whether an award amount is £0 (1) or greater than £0 (0), capturing zero-award records as a categorical signal.

However, as these records have no confirmed positive award amount, assigning them to a tier could misrepresent the target variable and distort model evaluation. A copy of the DataFrame is therefore created, excluding these two records from model training and evaluation while preserving them in the original dataset.

In [3]:
# Create a flag to identify pending/zero-award records
df_fe["pending_awards"] = (df_fe["award_amount"] == 0).astype(int)


# Check number of rows before filtering
print("Before filtering:", len(df_fe))


# Remove pending/zero-award records
df_fe = df_fe[df_fe["award_amount"] > 0].copy()


# Check number of rows after filtering
print("After filtering:", len(df_fe))


# Remove the temporary flag
df_fe = df_fe.drop(columns=["pending_awards"])


Before filtering: 5454
After filtering: 5452


</b>

## <u>**Award Amount Tiers**</u>

Apply Low, Medium and High classification tiers to reflect the variation in National Lottery Project Grants award amounts without transforming the target to fit a linear regression model.

The objective is not simply to create statistically balanced tiers. Although Random Forest combines multiple decision trees, substantial class imbalance can lead to majority-class predictions and weaker performance on minority classes. Class distribution must therefore be assessed alongside the interpretability of the tier boundaries.

The remaining 5,452 valid award records are transformed into three target classes using tertile-based thresholds, dividing the award amounts into approximately equal-sized groups. 

In [4]:
# Calculate the 33rd and 67th percentiles
p33 = df_fe['award_amount'].quantile(0.33)
p67 = df_fe['award_amount'].quantile(0.67)

print("33rd percentile:", p33)
print("67th percentile:", p67)

# Create the three award tiers
df_fe['award_tier'] = pd.cut(
    df_fe['award_amount'],
    bins=[-np.inf, float(p33), float(p67), np.inf],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True
)

# Check the number of observations in each tier
print(df_fe['award_tier'].value_counts())

# Check percentages
print(df_fe['award_tier'].value_counts(normalize=True) * 100)

33rd percentile: 23750.0
67th percentile: 30000.0
award_tier
Medium    1876
Low       1800
High      1776
Name: count, dtype: int64
award_tier
Medium    34.409391
Low       33.015407
High      32.575202
Name: proportion, dtype: float64


The 33rd percentile (£23,750) and 67th percentile (£30,000) were selected as tier boundaries. Awards of £23,750 or below are classified as Low, awards above £23,750 up to and including £30,000 as Medium, and awards above £30,000 as High. This produced a relatively balanced distribution across the Low (33.02%), Medium (34.41%) and High (32.58%) classes. No class is substantially underrepresented, providing a suitable starting point for training a Random Forest multi-class classifier. 

</b>

## <u>**Data Leakage**</u>

Columns that are post-outcome or used primarily for EDA are removed to reduce the risk of data leakage, which can inflate apparent model performance. The objective is to identify whether award tiers can be predicted from meaningful features available to the model, rather than variables that directly reveal the outcome.

Drop <code>award_amount</code>, <code>award_date</code>, <code>decision_month</code> and <code>award_year</code>. In addition to these columns drop <code>time_limited_priority</code> because the majority of awards are not time limited. 

In [5]:
# Drop columns that cause data leakage
df_fe = df_fe.drop(
    columns=[
        "award_amount",
        "award_date",
        "decision_quarter",
        "decision_month",
        "award_year",
        "time_limited_priority",
        "strand"
        ])


# Check remaining columns
df_fe.columns

Index(['ace_area', 'local_authority', 'main_discipline', 'award_tier'], dtype='str')

The <code>strand</code> variable was excluded from the predictive features because its categorical definitions include award-value thresholds. As the target variable is derived from <code>award_amount</code>, retaining strand could introduce target leakage and inflate apparent model performance.

In [6]:
# Check a sample of the model dataset
df_fe.sample(10)

,ace_area,local_authority,main_discipline,award_tier
954,London,Tower Hamlets,Dance,High
757,South East,Folkestone and Hythe,Music,Medium
84,London,Enfield,Theatre,High
3065,South East,Wokingham,Theatre,High
4710,London,Greenwich,Combined arts,High
2341,Midlands,Broxtowe,Combined arts,Medium
78,South East,Ashford,Combined arts,Low
3985,North,Salford,Music,Medium
3707,South West,Croydon,Visual arts,Low
1850,London,Lewisham,Literature,Low


</b>

## <u>**Save Model Data**</u>

A copy of the cleaned dataset is ready to train and test a Random Forest model with an <code>award_tier</code> target variable and <code>ace_area</code>, <code>local_authority</code> and <code>main dicipline</code>.

In [7]:
# Save feature-engineered cleaned data copy to a new CSV file
df_fe.to_csv("model_data.csv", index=False)

</b>

## <u>**Summary**</u>

Scaling is not applied because tree-based models, including Random Forest, do not require features to be scaled. Retaining the original values of numerical features preserves their interpretability for stakeholders.

Winsorisation is not applied because the skewed distribution reflects genuine variation in cultural funding rather than necessarily indicating data errors or noise. Capping extreme values could therefore obscure meaningful financial differences.

Data leakage is addressed by removing variables that directly reveal the target or may introduce information unavailable at prediction time, reducing the risk of artificially inflated model performance.

The <code>award_tier</code> variable is selected as the target to develop a supervised multi-class classification model aligned with the dataset and assessment requirements.